# Fase 7 - Ciclo ENSO com redes neurais ConvLSTM

Mesmo mecanismo das Fases 3 e 5, com uma ConvLSTM espaco-temporal que le sequencias de
campos do Pacifico equatorial (time, lat, lon, canais) e classifica as 4 fases; a
importancia por variavel/etapa vem de oclusao de canais.

Modulo: `src/nino_brasil/models/phase7_convlstm.py`. **Treino exige TensorFlow + GPU**
na maquina do usuario; a preparacao de dados (cubo, sequencias) roda sem TF.


In [1]:
import sys, pathlib
ROOT=pathlib.Path.cwd()
while not (ROOT/'src').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src')); sys.path.insert(0,str(ROOT))
import pandas as pd; from IPython.display import Image, display
STATS=ROOT/'data/processed/parquet/statistics'


In [2]:
import nino_brasil.models.phase7_convlstm as p7
import numpy as np
# Preparacao de dados (sem TF): cubo sintetico p/ demonstrar a API de sequencias.
cube=np.random.rand(120,8,24,3).astype('float32'); fase=np.random.randint(0,4,120)
seq,y=p7.make_sequences(cube, fase, seq_len=24)
print('sequencias:', seq.shape, '| alvo:', y.shape)
tr,va=p7.chronological_split(len(seq)); print('treino/val cronologico:', len(tr), len(va))


sequencias: (97, 24, 8, 24, 3) | alvo: (97,)
treino/val cronologico: 78 19


## Construcao da rede (requer TensorFlow)
```python
model = p7.build_convlstm_classifier(input_shape=seq.shape[1:], n_classes=4)
model.fit(seq[tr], y[tr], validation_data=(seq[va], y[va]), epochs=30)
imp = p7.channel_occlusion_importance(model, seq[va])  # XAI espacial
```
Fonte real do cubo: `data/processed/zarr/regridded/` via `p7.load_pacific_cube(...)`.
So avanca se vencer climatologia, persistencia, Fase 3 e Fase 5.
